## Figures for MS Thesis 


- Timeseries plot of DITN & LDEO temp, salinity, and velocity at various depths 
- Sliding window lag correlation analysis between DITN and LDEO (based on time series) 
- Timeseries plots of DITN history 
- T/S diagrams of DITN history 
- Timeseries plot of LDEO and mooring D, and sliding window lag correlation if makes sense 
- Timeseries plots of DITD and TNBD, sliding window lag correlation analysis 
- Timeseries and T/S diagrams for historical DITD
- T/S diagram for 2018 (TNBD, DITD)
- T/S diagram for DITD and D overlapping years (2017, 2019/20, 2021/22)
- T/S diagram for DITN, LDEO 2017, and D  


In [ ]:
import xarray as xr 
import numpy as np
import matplotlib.pyplot as plt
import scipy 
import gsw 
import glob
import pandas as pd 


## Data Availability 

Figure showing data availability for 2017 and 2018 moorings. Can add more if I go into historical data 

In [ ]:
# load in moorings 
# Load in datatree (DELL)
moorings = xr.open_datatree('/Users/mblan/Documents/research/TNB_tools/examples/TNB_Moorings_DataTree.nc')

# Load in datatree (MAC)
# moorings = xr.open_datatree('TNB_Moorings_DataTree.nc')

In [ ]:
# make a figure 
fig = plt.figure(figsize=(10,6))

# # define depth for y-axis 
# depth = np.arange(0,1600,100)

# # define time for x axis 
# years = np.arange('2017','2020',dtype='datetime64[Y]')
ds = moorings['D/2017/ctd/bot'].to_dataset()
# 2017 Data 
plt.plot(ds.time,np.ones(ds.time.size)*ds.z.values,label='D',color='tab:blue')
plt.plot(moorings['DITD/2017/ctd/bot'].time,np.ones(moorings['DITD/2017/ctd/bot'].time.size)*moorings['DITD/2017/ctd/bot'].z.values,label='DITD',color='tab:red')
plt.plot(moorings['DITN/2017/ctd/bot'].time,np.ones(moorings['DITN/2017/ctd/bot'].time.size)*moorings['DITN/2017/ctd/bot'].z.values,label='DITN',color='tab:orange')
plt.plot(moorings['DITS/2017/ctd/bot'].time,np.ones(moorings['DITS/2017/ctd/bot'].time.size)*moorings['DITS/2017/ctd/bot'].z.values,label='DITS',color='tab:green')
# plt.plot(moorings['LDEO/2017/ctd/bot'].time,np.ones(moorings['LDEO/2017/ctd/bot'].time.size)*moorings['LDEO/2017/ctd/bot'].z.values,label='LDEO',color='tab:purple')

# 2018 Data
plt.plot(moorings['DITD/2018/ctd/bot'].time,np.ones(moorings['DITD/2018/ctd/bot'].time.size)*moorings['DITD/2018/ctd/bot'].z.values,label='DITD',color='tab:red')
plt.plot(moorings['DITN/2018/ctd/bot'].time,np.ones(moorings['DITN/2018/ctd/bot'].time.size)*moorings['DITN/2018/ctd/bot'].z.values,label='DITN',color='tab:orange')
plt.plot(moorings['DITS/2018/ctd/bot'].time,np.ones(moorings['DITS/2018/ctd/bot'].time.size)*moorings['DITS/2018/ctd/bot'].z.values,label='DITS',color='tab:green')
plt.plot(moorings['TNBD/2018/ctd/bot'].time,np.ones(moorings['TNBD/2018/ctd/bot'].time.size)*moorings['TNBD/2018/ctd/bot'].z.values,label='TNBD',color='tab:purple')

plt.ylim(1500,0)
plt.xlim(datetime(2017,1,1),datetime(2020,1,1))
plt.grid()
plt.ylabel('Depth (m)')
plt.xlabel('Time')

# Remove duplicates
handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
plt.legend(by_label.values(), by_label.keys(), loc='upper right')

plt.title('Data Availability')


## Previous work 

In [ ]:
def summarize_all(dt):
    """Return a DataFrame summarizing the entire DataTree."""
    import pandas as pd
    import numpy as np

    rows = []

    for mooring, mooring_node in dt.children.items():
        for year, year_node in mooring_node.children.items():
            for instrument, instr_node in year_node.children.items():
                for position, pos_node in instr_node.children.items():

                    ds = pos_node.ds

                    if "time" in ds.coords:
                        t = ds["time"].to_numpy()
                        t_start = t[0]
                        t_end = t[-1]
                        dt_med = np.median(np.diff(t)) if len(t) > 1 else np.nan
                    else:
                        t_start = t_end = dt_med = np.nan

                    depth = ds.coords["z"].item() if "z" in ds.coords else np.nan

                    rows.append({
                        "mooring": mooring,
                        "year": year,
                        "instrument": instrument,
                        "position": position,
                        "depth": depth,
                        "variables": list(ds.data_vars),
                        "time_start": t_start,
                        "time_end": t_end,
                        "sampling_period": dt_med,
                        "n_points": ds.sizes.get("time", np.nan),
                    })

    df = pd.DataFrame(rows)
    df = df.sort_values(["mooring", "year", "instrument", "position"])
    return df.reset_index(drop=True)


In [ ]:
def plot_timeseries(years,var,mooring,instrument,position,color,label):
    """
    inputs:
    years: years to loop over (array containing years in string format)
    var: variable of interest (string)
    mooring: mooring of interest (string)
    instrument: instrument of interest (string)
    position: position of interest (string)
    color: color to use for line plotting (string)
    """
    for i in range(len(years)):
        var = var
        ds = dt[mooring][years[i]][instrument][position]
        if var in ds.data_vars:
            ds[var].plot(color=color,label=label)

In [ ]:
# load in individual datasets for now for each mooring - figure out datatree structure later 
dt = xr.open_datatree('TNB_mooring_datatree.nc')

In [ ]:
df = summarize_all(dt)

In [ ]:
df

### Load Data 

Data to load: 

- TNBD top and bottom sensors (ctd + velocity)
- DITD (full dataset)
- DITN (full dataset)
- LDEO (full dataset)
- D (full dataset)

#### Load LDEO data 

In [ ]:
LDEO_ctd = xr.open_dataset('/Users/mblan/Documents/TNB_Research/data/LDEO_TNB_mooring_data_QC/LDEO_TNB_temperature_salinity.nc')
LDEO_vel_top = xr.open_dataset('/Users/mblan/Documents/TNB_Research/data/LDEO_TNB_mooring_data_QC/LDEO_TNB_230m_current_velocity.nc')
LDEO_vel_mid = xr.open_dataset('/Users/mblan/Documents/TNB_Research/data/LDEO_TNB_mooring_data_QC/LDEO_TNB_300m_current_velocity.nc')
LDEO_vel_bot = xr.open_dataset('/Users/mblan/Documents/TNB_Research/data/LDEO_TNB_mooring_data_QC/LDEO_TNB_370m_current_velocity.nc')
LDEO_awac = xr.open_dataset('/Users/mblan/Documents/TNB_Research/data/LDEO_TNB_mooring_data_QC/LDEO_TNB_40m_surface_current_profiles.nc')

In [ ]:
LDEO_datasets = [
    LDEO_ctd,
    LDEO_vel_top,
    LDEO_vel_mid,
    LDEO_vel_bot,
    LDEO_awac
]

#### load DITN data 

In [ ]:
df.query("mooring == 'P3'")

In [ ]:
# load in all datasets for DITD mooring 
paths = glob.glob("//thepenguin/penguin2/Data/TNB/mooring/DITx/DITN/*.nc")

In [ ]:
paths

In [ ]:
# create a list to hold datasets 
ds_list = []
for path in paths:
    ds = xr.open_dataset(path)
    ds_list.append(ds)

In [ ]:
for i in range(len(ds_list)):
    print(ds_list[i].attrs['description'] + ', depth= ' + ds_list[i].z.values.astype(str))
# print(ds_list[1].attrs['description'] + ', depth= ' + ds_list[1].z.values.astype(str))

In [ ]:
# sepearte mooring data by top, mid, bottom sensors 
DITN_2014_speed_top = ds_list[0].sel(z=75).rename({'DateTime':'time'}).assign_coords({'lat':-75.3602,'lon':164.7495})
DITN_2014_speed_mid = ds_list[0].sel(z=276).rename({'DateTime':'time'}).assign_coords({'lat':-75.3602,'lon':164.7495})
DITN_2014_speed_bot = ds_list[0].sel(z=674).rename({'DateTime':'time'}).assign_coords({'lat':-75.3602,'lon':164.7495})

DITN_2014_ctd_top = ds_list[1].sel(z=77).rename({'DateTime':'time'}).assign_coords({'lat':-75.3602,'lon':164.7495})
DITN_2014_ctd_mid = ds_list[1].sel(z=278).rename({'DateTime':'time'}).assign_coords({'lat':-75.3602,'lon':164.7495})
DITN_2014_ctd_bot = ds_list[1].sel(z=667).rename({'DateTime':'time'}).assign_coords({'lat':-75.3602,'lon':164.7495})

DITN_2014_temp_top = ds_list[2].sel(z=108).rename({'DateTime':'time'}).assign_coords({'lat':-75.3602,'lon':164.7495})
DITN_2014_temp_mid1 = ds_list[2].sel(z=148).rename({'DateTime':'time'}).assign_coords({'lat':-75.3602,'lon':164.7495})
DITN_2014_temp_mid2 = ds_list[2].sel(z=188).rename({'DateTime':'time'}).assign_coords({'lat':-75.3602,'lon':164.7495})
DITN_2014_temp_bot = ds_list[2].sel(z=228).rename({'DateTime':'time'}).assign_coords({'lat':-75.3602,'lon':164.7495})

DITN_2015_ctd_top = ds_list[3].sel(z=73).rename({'DateTime':'time'}).assign_coords({'lat':-75.3601,'lon':164.7486})
DITN_2015_ctd_mid = ds_list[3].sel(z=275).rename({'DateTime':'time'}).assign_coords({'lat':-75.3601,'lon':164.7486})
DITN_2015_ctd_bot = ds_list[3].sel(z=666).rename({'DateTime':'time'}).assign_coords({'lat':-75.3601,'lon':164.7486})

DITN_2015_speed_top = ds_list[4].sel(z=71).rename({'DateTime':'time'}).assign_coords({'lat':-75.3601,'lon':164.7486})
DITN_2015_speed_mid = ds_list[4].sel(z=273).rename({'DateTime':'time'}).assign_coords({'lat':-75.3601,'lon':164.7486})
DITN_2015_speed_bot = ds_list[4].sel(z=667).rename({'DateTime':'time'}).assign_coords({'lat':-75.3601,'lon':164.7486})

DITN_2015_temp_top = ds_list[5].sel(z=113).rename({'DateTime':'time'}).assign_coords({'lat':-75.3601,'lon':164.7486})
DITN_2015_temp_mid1 = ds_list[5].sel(z=153).rename({'DateTime':'time'}).assign_coords({'lat':-75.3601,'lon':164.7486})
DITN_2015_temp_mid2 = ds_list[5].sel(z=193).rename({'DateTime':'time'}).assign_coords({'lat':-75.3601,'lon':164.7486})
DITN_2015_temp_bot = ds_list[5].sel(z=233).rename({'DateTime':'time'}).assign_coords({'lat':-75.3601,'lon':164.7486})

DITN_2017_vel_top = ds_list[6].sel(z=76).rename({'DateTime':'time'}).assign_coords({'lat':-75.3608,'lon':164.7465})
DITN_2017_vel_mid = ds_list[6].sel(z=271).rename({'DateTime':'time'}).assign_coords({'lat':-75.3608,'lon':164.7465})
DITN_2017_vel_bot = ds_list[6].sel(z=694).rename({'DateTime':'time'}).assign_coords({'lat':-75.3608,'lon':164.7465})

DITN_2017_speed_top = ds_list[7].sel(z=75).rename({'DateTime':'time'}).assign_coords({'lat':-75.3608,'lon':164.7465})
DITN_2017_speed_mid = ds_list[7].sel(z=270).rename({'DateTime':'time'}).assign_coords({'lat':-75.3608,'lon':164.7465})
DITN_2017_speed_bot = ds_list[7].sel(z=667).rename({'DateTime':'time'}).assign_coords({'lat':-75.3608,'lon':164.7465})

DITN_2017_ctd_top = ds_list[8].sel(z=75).rename({'DateTime':'time'}).assign_coords({'lat':-75.3608,'lon':164.7465})
DITN_2017_ctd_mid = ds_list[8].sel(z=275).rename({'DateTime':'time'}).assign_coords({'lat':-75.3608,'lon':164.7465})
DITN_2017_ctd_bot = ds_list[8].sel(z=675).rename({'DateTime':'time'}).assign_coords({'lat':-75.3608,'lon':164.7465})

DITN_2017_temp_top = ds_list[9].sel(z=197).rename({'DateTime':'time'}).assign_coords({'lat':-75.3608,'lon':164.7465})
DITN_2017_temp_mid = ds_list[9].sel(z=330).rename({'DateTime':'time'}).assign_coords({'lat':-75.3608,'lon':164.7465})
DITN_2017_temp_bot = ds_list[9].sel(z=360).rename({'DateTime':'time'}).assign_coords({'lat':-75.3608,'lon':164.7465})

DITN_2018_ctd_top = ds_list[10].sel(z=73).rename({'DateTime':'time'}).assign_coords({'lat':-75.361,'lon':164.7430})
DITN_2018_ctd_mid = ds_list[10].sel(z=274).rename({'DateTime':'time'}).assign_coords({'lat':-75.361,'lon':164.7430})
DITN_2018_ctd_bot = ds_list[10].sel(z=670).rename({'DateTime':'time'}).assign_coords({'lat':-75.361,'lon':164.7430})

DITN_2018_temp_top = ds_list[11].sel(z=114).rename({'DateTime':'time'}).assign_coords({'lat':-75.361,'lon':164.7430})
DITN_2018_temp_mid1 = ds_list[11].sel(z=154).rename({'DateTime':'time'}).assign_coords({'lat':-75.361,'lon':164.7430})
DITN_2018_temp_mid2 = ds_list[11].sel(z=194).rename({'DateTime':'time'}).assign_coords({'lat':-75.361,'lon':164.7430})
DITN_2018_temp_bot = ds_list[11].sel(z=224).rename({'DateTime':'time'}).assign_coords({'lat':-75.361,'lon':164.7430})

DITN_2018_vel_top = ds_list[13].sel(z=71).rename({'DateTime':'time'}).assign_coords({'lat':-75.361,'lon':164.7430})
DITN_2018_vel_mid = ds_list[13].sel(z=263).rename({'DateTime':'time'}).assign_coords({'lat':-75.361,'lon':164.7430})
DITN_2018_vel_bot = ds_list[13].sel(z=666).rename({'DateTime':'time'}).assign_coords({'lat':-75.361,'lon':164.7430})

DITN_2019_vel_top = ds_list[12].sel(z=67).rename({'DateTime':'time'}).assign_coords({'lat':-75.3486,'lon':164.8159})
DITN_2019_vel_mid = ds_list[12].sel(z=255).rename({'DateTime':'time'}).assign_coords({'lat':-75.3486,'lon':164.8159})
DITN_2019_vel_bot = ds_list[12].sel(z=649).rename({'DateTime':'time'}).assign_coords({'lat':-75.3486,'lon':164.8159})

DITN_2019_ctd_top = ds_list[14].sel(z=71).rename({'DateTime':'time'}).assign_coords({'lat':-75.3486,'lon':164.8159})
DITN_2019_ctd_mid = ds_list[14].sel(z=271).rename({'DateTime':'time'}).assign_coords({'lat':-75.3486,'lon':164.8159})

DITN_2019_temp_top = ds_list[15].sel(z=112).rename({'DateTime':'time'}).assign_coords({'lat':-75.3486,'lon':164.8159})
DITN_2019_temp_mid1 = ds_list[15].sel(z=152).rename({'DateTime':'time'}).assign_coords({'lat':-75.3486,'lon':164.8159})
DITN_2019_temp_mid2 = ds_list[15].sel(z=192).rename({'DateTime':'time'}).assign_coords({'lat':-75.3486,'lon':164.8159})
DITN_2019_temp_bot = ds_list[15].sel(z=232).rename({'DateTime':'time'}).assign_coords({'lat':-75.3486,'lon':164.8159})

DITN_2020_vel_top = ds_list[16].sel(z=54).rename({'DateTime':'time'}).assign_coords({'lat':-75.3493,'lon':164.8170})
DITN_2020_vel_mid = ds_list[16].sel(z=252).rename({'DateTime':'time'}).assign_coords({'lat':-75.3493,'lon':164.8170})
DITN_2020_vel_bot = ds_list[16].sel(z=671).rename({'DateTime':'time'}).assign_coords({'lat':-75.3493,'lon':164.8170})

DITN_2020_ctd_mid = ds_list[17].sel(z=269).rename({'DateTime':'time'}).assign_coords({'lat':-75.3493,'lon':164.8170})
DITN_2020_ctd_bot = ds_list[17].sel(z=667).rename({'DateTime':'time'}).assign_coords({'lat':-75.3493,'lon':164.8170})

DITN_2020_temp_top = ds_list[18].sel(z=95).rename({'DateTime':'time'}).assign_coords({'lat':-75.3493,'lon':164.8170})
DITN_2020_temp_mid1 = ds_list[18].sel(z=135).rename({'DateTime':'time'}).assign_coords({'lat':-75.3493,'lon':164.8170})
DITN_2020_temp_mid2 = ds_list[18].sel(z=175).rename({'DateTime':'time'}).assign_coords({'lat':-75.3493,'lon':164.8170})
DITN_2020_temp_bot = ds_list[18].sel(z=215).rename({'DateTime':'time'}).assign_coords({'lat':-75.3493,'lon':164.8170})



In [ ]:
DITN_datasets = [
    DITN_2014_speed_top,
    DITN_2014_speed_mid,
    DITN_2014_speed_bot,
    DITN_2014_ctd_top,
    DITN_2014_ctd_mid,
    DITN_2014_ctd_bot,
    DITN_2014_temp_top,
    DITN_2014_temp_mid1,
    DITN_2014_temp_mid2,
    DITN_2014_temp_bot,
    DITN_2015_ctd_top,
    DITN_2015_ctd_mid,
    DITN_2015_ctd_bot,
    DITN_2015_speed_top,
    DITN_2015_speed_mid,
    DITN_2015_speed_bot,
    DITN_2015_temp_top,
    DITN_2015_temp_mid1,
    DITN_2015_temp_mid2,
    DITN_2015_temp_bot,
    DITN_2017_vel_top,
    DITN_2017_vel_mid,
    DITN_2017_vel_bot,
    DITN_2017_speed_top,
    DITN_2017_speed_mid,
    DITN_2017_speed_bot,
    DITN_2017_ctd_top,
    DITN_2017_ctd_mid,
    DITN_2017_ctd_bot,
    DITN_2017_temp_top,
    DITN_2017_temp_mid,
    DITN_2017_temp_bot,
    DITN_2018_ctd_top,
    DITN_2018_ctd_mid,
    DITN_2018_ctd_bot,
    DITN_2018_temp_top,
    DITN_2018_temp_mid1,
    DITN_2018_temp_mid2,
    DITN_2018_temp_bot,
    DITN_2018_vel_top,
    DITN_2018_vel_mid,
    DITN_2018_vel_bot,
    DITN_2019_vel_top,
    DITN_2019_vel_mid,
    DITN_2019_vel_bot,
    DITN_2019_ctd_top,
    DITN_2019_ctd_mid,
    DITN_2019_temp_top,
]

#### Load in DITD data 

In [ ]:
# load in all datasets for DITD mooring 
paths = glob.glob("//thepenguin/penguin2/Data/TNB/mooring/DITx/DITD/*.nc")

In [ ]:
paths

In [ ]:
paths[0]

In [ ]:
# create a list to hold datasets 
ds_list = []
for path in paths:
    ds = xr.open_dataset(path)
    ds_list.append(ds)

In [ ]:
for i in range(len(ds_list)):
    print(ds_list[i].attrs['description'] + ', depth= ' + ds_list[i].z.values.astype(str))
# print(ds_list[1].attrs['description'] + ', depth= ' + ds_list[1].z.values.astype(str))

In [ ]:
DITD_2017_vel_bot = ds_list[0].squeeze().rename({'DateTime':'time'}).assign_coords({'lat':-75.2757,'lon':164.0673})
DITD_2017_ctd_bot = ds_list[1].squeeze().rename({'DateTime':'time'}).assign_coords({'lat':-75.2757,'lon':164.0673})
DITD_2019_vel_bot = ds_list[4].squeeze().rename({'DateTime':'time'}).assign_coords({'lat':-75.2766,'lon':164.0700})
DITD_2019_ctd_bot = ds_list[5].squeeze().rename({'DateTime':'time'}).assign_coords({'lat':-75.2766,'lon':164.0700})
DITD_2020_vel_bot = ds_list[6].squeeze().rename({'DateTime':'time'}).assign_coords({'lat':-75.2780,'lon':164.0603})
DITD_2020_ctd_bot = ds_list[7].squeeze().rename({'DateTime':'time'}).assign_coords({'lat':-75.2780,'lon':164.0603})
DITD_2022_vel_bot = ds_list[8].squeeze().rename({'DateTime':'time'}).assign_coords({'lat':-75.2778,'lon':164.0607})
DITD_2022_ctd_bot = ds_list[9].squeeze().rename({'DateTime':'time'}).assign_coords({'lat':-75.2778,'lon':164.0607})
DITD_2024_vel_bot = ds_list[10].squeeze().rename({'DateTime':'time'}).assign_coords({'lat':-75.2777,'lon':164.0602})
DITD_2024_ctd_bot = ds_list[11].squeeze().rename({'DateTime':'time'}).assign_coords({'lat':-75.2777,'lon':164.0602})
# seperate dataset 1803 velocity and sbe into top and bottom components 
DITD_2018_vel_top = ds_list[2].sel(z=1145).rename({'DateTime':'time'}).assign_coords({'lat':-75.2773,'lon':164.071}) # bottom velocity
DITD_2018_vel_bot = ds_list[2].sel(z=1244).rename({'DateTime':'time'}).assign_coords({'lat':-75.2773,'lon':164.071}) # top velocity

DITD_2018_ctd_top = ds_list[3].sel(z=1122).rename({'DateTime':'time'}).assign_coords({'lat':-75.2773,'lon':164.071})  # bottom sbe
DITD_2018_ctd_bot = ds_list[3].sel(z=1218).rename({'DateTime':'time'}).assign_coords({'lat':-75.2773,'lon':164.071})  # top sbe

In [ ]:
fig = plt.figure()
DITD_2024_ctd_bot.S.plot()

In [ ]:
# load in DITD 2018 data - data from online is missing 
# DITD_2018_ctd_top = xr.open_dataset('/Users/mblan/Documents/TNB_Research/data/DITD_2018_2019/sbe_1100.nc')
# DITD_2018_ctd_bot = xr.open_dataset('/Users/mblan/Documents/TNB_Research/data/DITD_2018_2019/sbe_1200.nc')

# DITD_2018_vel_top = xr.open_dataset('/Users/mblan/Documents/TNB_Research/data/DITD_2018_2019/vel_1100.nc')
# DITD_2018_vel_bot = xr.open_dataset('/Users/mblan/Documents/TNB_Research/data/DITD_2018_2019/vel_1200.nc')

In [ ]:
# FIX TIME ON DITD veloctiy top sensor 
# DITD_2018_vel_top['time'] = DITD_2018_vel_top.time - pd.Timedelta(12,'h')

In [ ]:
DITD_datasets = [
    DITD_2017_vel_bot,
    DITD_2017_ctd_bot,
    DITD_2018_vel_top,
    DITD_2018_vel_bot,
    DITD_2018_ctd_top,
    DITD_2018_ctd_bot,
    DITD_2019_vel_bot,
    DITD_2019_ctd_bot,
    DITD_2020_vel_bot,
    DITD_2020_ctd_bot,
    DITD_2022_vel_bot,
    DITD_2022_ctd_bot,
    DITD_2024_vel_bot,
    DITD_2024_ctd_bot
]

#### Load TNBD data /

In [ ]:
df.query("mooring == 'TNBD' ")

In [ ]:
TNBD_ctd_top = dt['TNBD']['2018']['ctd']['top'].to_dataset()
TNBD_ctd_bot = dt['TNBD']['2018']['ctd']['bot'].to_dataset()
TNBD_vel_top = dt['TNBD']['2018']['vel']['top'].to_dataset()
TNBD_vel_bot = dt['TNBD']['2018']['vel']['bot'].to_dataset()

In [ ]:
TNBD_datasets = [
    TNBD_ctd_top,
    TNBD_ctd_bot,
    TNBD_vel_top,
    TNBD_vel_bot
]

### Correct LDEO, DITN, and DITD velocity data for magnetic declination 

In [ ]:
def rotate_velocity(u,v,angle):
    """Function that rotates velocity component vectors using rotation equations
       
       Inputs: v: u velocity, v velocity, angle: angle between vectors [radians]
       Outputs: rotated v velocity vectors 
    """
    u_prime = u*np.cos(angle) - v*np.sin(angle)
    v_prime = u*np.sin(angle) + v*np.cos(angle)
    
    return u_prime,v_prime

In [ ]:
# example on how to correct using above function 
# correct velocity data for magnetic declination 
# magnetic declination for DITD mooring location: 136.398985904
# mag_correction_degrees = 136.39896
# mag_correction_radians = np.deg2rad(mag_correction_degrees)
# u_prime_1100,v_prime_1100 = rotate_velocity(vel_1100.east,vel_1100.north,-mag_correction_radians)
# u_prime_1200,v_prime_1200 = rotate_velocity(vel_1200.east,vel_1200.north,-mag_correction_radians)

#### LDEO correction 

magnetic delcination angle = 135.37

In [ ]:
mag_correction_degrees = 135.37
mag_correction_radians = np.deg2rad(mag_correction_degrees)

In [ ]:
# top sensor 
u_prime, v_prime = rotate_velocity(LDEO_vel_top.east,LDEO_vel_top.north,-mag_correction_radians)

LDEO_vel_top['u'] = u_prime
LDEO_vel_top['v'] = v_prime

In [ ]:
# mid sensor 
u_prime, v_prime = rotate_velocity(LDEO_vel_mid.east,LDEO_vel_mid.north,-mag_correction_radians)

LDEO_vel_mid['u'] = u_prime
LDEO_vel_mid['v'] = v_prime

In [ ]:
# bot sensor 
u_prime, v_prime = rotate_velocity(LDEO_vel_bot.east,LDEO_vel_bot.north,-mag_correction_radians)

LDEO_vel_bot['u'] = u_prime
LDEO_vel_bot['v'] = v_prime

In [ ]:
# awac sensor 
u_prime, v_prime = rotate_velocity(LDEO_awac.east,LDEO_awac.north,-mag_correction_radians)

LDEO_awac['u'] = u_prime
LDEO_awac['v'] = v_prime

#### DITD Correction SKIP FOR NOW 

magnetic delcination angle = 136.46

Note: correcting all datasets for all years using this angle for now - need to go back and get magnetic declination angle for each year to be more precise since the angle does change yearly (slight change)


In [ ]:
# DITD_datasets = [
#     DITD_2017_vel_bot,
#     DITD_2017_ctd_bot,
#     DITD_2018_vel_top,
#     DITD_2018_vel_bot,
#     DITD_2018_ctd_top,
#     DITD_2018_ctd_bot,
#     DITD_2019_vel_bot,
#     DITD_2019_ctd_bot,
#     DITD_2020_vel_bot,
#     DITD_2020_ctd_bot,
#     DITD_2022_vel_bot,
#     DITD_2022_ctd_bot,
#     DITD_2024_vel_bot,
#     DITD_2024_ctd_bot
# ]

In [ ]:
mag_correction_degrees = 136.46
mag_correction_radians = np.deg2rad(mag_correction_degrees)

In [ ]:
# 2018 is already corrected for 
# 2017
u_prime, v_prime = rotate_velocity(DITD_2017_vel_bot.u,DITD_2017_vel_bot.v,-mag_correction_radians)

DITD_2017_vel_bot['u'] = u_prime
DITD_2017_vel_bot['v'] = v_prime

In [ ]:
u_prime, v_prime = rotate_velocity(DITD_2018_vel_bot.u,DITD_2018_vel_bot.v,-mag_correction_radians)

DITD_2018_vel_bot['u'] = u_prime
DITD_2018_vel_bot['v'] = v_prime

In [ ]:
# 2019
u_prime, v_prime = rotate_velocity(DITD_2019_vel_bot.u,DITD_2019_vel_bot.v,-mag_correction_radians)

DITD_2019_vel_bot['u'] = u_prime
DITD_2019_vel_bot['v'] = v_prime

In [ ]:
# 2020
u_prime, v_prime = rotate_velocity(DITD_2020_vel_bot.u,DITD_2020_vel_bot.v,-mag_correction_radians)

DITD_2020_vel_bot['u'] = u_prime
DITD_2020_vel_bot['v'] = v_prime

In [ ]:
# 2022
u_prime, v_prime = rotate_velocity(DITD_2022_vel_bot.u,DITD_2022_vel_bot.v,-mag_correction_radians)

DITD_2022_vel_bot['u'] = u_prime
DITD_2022_vel_bot['v'] = v_prime

In [ ]:
# 2024
u_prime, v_prime = rotate_velocity(DITD_2024_vel_bot.u,DITD_2024_vel_bot.v,-mag_correction_radians)

DITD_2024_vel_bot['u'] = u_prime
DITD_2024_vel_bot['v'] = v_prime

In [ ]:
DITD_datasets

#### DITN correction SKIP FOR NOW 

magnetic declination angle = 135.76

In [ ]:
DITN_datasets = [
    DITN_2014_speed_top,
    DITN_2014_speed_mid,
    DITN_2014_speed_bot,
    DITN_2014_ctd_top,
    DITN_2014_ctd_mid,
    DITN_2014_ctd_bot,
    DITN_2014_temp_top,
    DITN_2014_temp_mid1,
    DITN_2014_temp_mid2,
    DITN_2014_temp_bot,
    DITN_2015_ctd_top,
    DITN_2015_ctd_mid,
    DITN_2015_ctd_bot,
    DITN_2015_speed_top,
    DITN_2015_speed_mid,
    DITN_2015_speed_bot,
    DITN_2015_temp_top,
    DITN_2015_temp_mid1,
    DITN_2015_temp_mid2,
    DITN_2015_temp_bot,
    DITN_2017_vel_top,
    DITN_2017_vel_mid,
    DITN_2017_vel_bot,
    DITN_2017_speed_top,
    DITN_2017_speed_mid,
    DITN_2017_speed_bot,
    DITN_2017_ctd_top,
    DITN_2017_ctd_mid,
    DITN_2017_ctd_bot,
    DITN_2017_temp_top,
    DITN_2017_temp_mid,
    DITN_2017_temp_bot,
    DITN_2018_ctd_top,
    DITN_2018_ctd_mid,
    DITN_2018_ctd_bot,
    DITN_2018_temp_top,
    DITN_2018_temp_mid1,
    DITN_2018_temp_mid2,
    DITN_2018_temp_bot,
    DITN_2018_vel_top,
    DITN_2018_vel_mid,
    DITN_2018_vel_bot,
    DITN_2019_vel_top,
    DITN_2019_vel_mid,
    DITN_2019_vel_bot,
    DITN_2019_ctd_top,
    DITN_2019_ctd_mid,
    DITN_2019_temp_top,
]

In [ ]:
DITN_2017_vel_top

In [ ]:
mag_correction_degrees = 135.76
mag_correction_radians = np.deg2rad(mag_correction_degrees)

In [ ]:
# 2018 is already corrected for 
# 2017
u_prime, v_prime = rotate_velocity(DITN_2017_vel_top.u,DITN_2017_vel_top.v,-mag_correction_radians)

DITN_2017_vel_top['u'] = u_prime
DITN_2017_vel_top['v'] = v_prime

u_prime, v_prime = rotate_velocity(DITN_2017_vel_mid.u,DITN_2017_vel_mid.v,-mag_correction_radians)

DITN_2017_vel_mid['u'] = u_prime
DITN_2017_vel_mid['v'] = v_prime

u_prime, v_prime = rotate_velocity(DITN_2017_vel_bot.u,DITN_2017_vel_bot.v,-mag_correction_radians)

DITN_2017_vel_bot['u'] = u_prime
DITN_2017_vel_bot['v'] = v_prime

In [ ]:
# 2018 is already corrected for 
# 2017
u_prime, v_prime = rotate_velocity(DITN_2018_vel_top.u,DITN_2018_vel_top.v,-mag_correction_radians)

DITN_2018_vel_top['u'] = u_prime
DITN_2018_vel_top['v'] = v_prime

u_prime, v_prime = rotate_velocity(DITN_2018_vel_mid.u,DITN_2018_vel_mid.v,-mag_correction_radians)

DITN_2018_vel_mid['u'] = u_prime
DITN_2018_vel_mid['v'] = v_prime

u_prime, v_prime = rotate_velocity(DITN_2018_vel_bot.u,DITN_2018_vel_bot.v,-mag_correction_radians)

DITN_2018_vel_bot['u'] = u_prime
DITN_2018_vel_bot['v'] = v_prime

In [ ]:
# 2018 is already corrected for 
# 2019
u_prime, v_prime = rotate_velocity(DITN_2019_vel_top.u,DITN_2019_vel_top.v,-mag_correction_radians)

DITN_2019_vel_top['u'] = u_prime
DITN_2019_vel_top['v'] = v_prime

u_prime, v_prime = rotate_velocity(DITN_2019_vel_mid.u,DITN_2019_vel_mid.v,-mag_correction_radians)

DITN_2019_vel_mid['u'] = u_prime
DITN_2019_vel_mid['v'] = v_prime

u_prime, v_prime = rotate_velocity(DITN_2019_vel_bot.u,DITN_2019_vel_bot.v,-mag_correction_radians)

DITN_2019_vel_bot['u'] = u_prime
DITN_2019_vel_bot['v'] = v_prime

### DITN vs. LDEO 2017/2018

DITN_2018_ctd_top

In [ ]:
DITN_2017_ctd_top.z.values, DITN_2017_ctd_mid.z.values, DITN_2017_ctd_bot.z.values

In [ ]:
LDEO_ctd_bot = LDEO_ctd.sel(depth=360)

In [ ]:
LDEO_ctd_bot

In [ ]:
DITN_2018_ctd_mid

In [ ]:
%matplotlib widget

In [ ]:
fig = plt.figure(figsize=(12,4))
ds = DITN_2017_ctd_top
ds1 = DITN_2017_ctd_mid
ds2 = DITN_2017_ctd_bot

plt.plot(ds.time,ds.S,label='75 m')
plt.plot(ds1.time,ds1.S,label='275 m')
plt.plot(ds2.time,ds2.S,label='675 m')

plt.legend()
plt.ylabel('Practical Salinity')
plt.title('DITN 2017/2018 Salinity')

In [ ]:
LDEO_ctd.depth

In [ ]:
fig = plt.figure(figsize=(12,4))
ds = LDEO_ctd

plt.plot(ds.time,ds.sel(depth=47).practical_salinity,label='47 m')
plt.plot(ds.time,ds.sel(depth=95).practical_salinity,label='95 m')
plt.plot(ds.time,ds.sel(depth=119).practical_salinity,label='119 m')
plt.plot(ds.time,ds.sel(depth=144).practical_salinity,label='144 m')
plt.plot(ds.time,ds.sel(depth=168).practical_salinity,label='168 m')
plt.plot(ds.time,ds.sel(depth=219).practical_salinity,label='219 m')
plt.plot(ds.time,ds.sel(depth=360).practical_salinity,label='360 m')



plt.legend()
plt.ylabel('Practical Salinity')
plt.title('LDEO 2017/2018 Salinity')

In [ ]:
# time series plots 
ds = DITN_2017_ctd_bot
ds1 = LDEO_ctd_bot
ds2 = DITN_2017_ctd_mid
fig = plt.figure(figsize=(12,4))
plt.plot(ds.time,ds.S,label='DITN 675 m')
plt.plot(ds2.time,ds2.S,label='DITN 275 m')
plt.plot(ds1.time,ds1.practical_salinity,label='LDEO 360 m')

plt.legend()
plt.ylabel('Practical Salinity')
plt.title('LDEO and DITN Bottom Salinity')

In [ ]:
# create merged DITN mid and bottom sensor datasets 
df.query("mooring == 'DITN' & position == 'mid' & instrument == 'ctd'")

##### Resample DITN data to create one continuous dataset

In [ ]:
# need to resample 2017, 2018, and 2020 to 10 minutes 
DITN_2017_ctd_bot = DITN_2017_ctd_bot.resample(time='10min').mean()
DITN_2018_ctd_bot = DITN_2018_ctd_bot.resample(time='10min').mean()
DITN_2020_ctd_bot = DITN_2020_ctd_bot.resample(time='10min').mean()

In [ ]:
# merge datasets together 
DITN_ctd_bot = xr.concat([DITN_2014_ctd_bot,DITN_2015_ctd_bot,DITN_2017_ctd_bot,DITN_2018_ctd_bot,DITN_2020_ctd_bot],dim='time')

# make full time index to fill in with nans where time is missing - aids in plotting later 
full_time = pd.date_range(
    start="2014-12-12T14:00:00",
    end="2022-03-15T13:50:00",
    freq="10min"
)

DITN_ctd_bot = DITN_ctd_bot.sortby("time")     # VERY important after concatenation
DITN_ctd_bot = DITN_ctd_bot.reindex(time=full_time)



In [ ]:
# DITN mid
DITN_2017_ctd_mid = DITN_2017_ctd_mid.resample(time='10min').mean()
DITN_2018_ctd_mid = DITN_2018_ctd_mid.resample(time='10min').mean()
DITN_2019_ctd_mid = DITN_2019_ctd_mid.resample(time='10min').mean()
DITN_2020_ctd_mid = DITN_2020_ctd_mid.resample(time='10min').mean()

In [ ]:
# merge datasets together 
DITN_ctd_mid = xr.concat([DITN_2014_ctd_mid,DITN_2015_ctd_mid,DITN_2017_ctd_mid,DITN_2018_ctd_mid,DITN_2019_ctd_mid,DITN_2020_ctd_mid],dim='time')

In [ ]:
DITN_ctd_mid.time[-1]

In [ ]:
# make full time index to fill in with nans where time is missing - aids in plotting later 
full_time = pd.date_range(
    start="2014-12-12T14:00:00",
    end="2022-03-15T13:50:00",
    freq="10min"
)

DITN_ctd_mid = DITN_ctd_mid.sortby("time")     # VERY important after concatenation
DITN_ctd_mid = DITN_ctd_mid.reindex(time=full_time)

In [ ]:
# DITN top 
DITN_2017_ctd_top = DITN_2017_ctd_top.resample(time='10min').mean()
DITN_2018_ctd_top = DITN_2018_ctd_top.resample(time='10min').mean()
DITN_2019_ctd_top = DITN_2019_ctd_top.resample(time='10min').mean()

In [ ]:
DITN_ctd_top = xr.concat([DITN_2014_ctd_top,DITN_2015_ctd_top,DITN_2017_ctd_top,DITN_2018_ctd_top,DITN_2019_ctd_top],dim='time')

In [ ]:
DITN_ctd_top.time[-1]

In [ ]:
# make full time index to fill in with nans where time is missing - aids in plotting later 
full_time = pd.date_range(
    start="2014-12-12T14:00:00",
    end="2020-03-17T05:50:00",
    freq="10min"
)

DITN_ctd_top = DITN_ctd_top.sortby("time")     # VERY important after concatenation
DITN_ctd_top = DITN_ctd_top.reindex(time=full_time)

In [ ]:
fig = plt.figure(figsize=(12,4))
plt.plot(DITN_ctd_top.time,DITN_ctd_top.S,label='75 m')
plt.plot(DITN_ctd_mid.time,DITN_ctd_mid.S,label='275 m')
plt.plot(DITN_ctd_bot.time,DITN_ctd_bot.S,label='670 m')
plt.legend()
plt.ylabel('Practical Salinity')
plt.title('DITN Salinity')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import gsw

# Extract variables
temp = DITN_ctd_bot.T
sal  = DITN_ctd_bot.S
time = pd.to_datetime(DITN_ctd_bot.time.values)
years = time.year
unique_years = np.unique(years)

# ----- Density Grid -----
mint = temp.min()
maxt = temp.max()
mins = sal.min()
maxs = sal.max()

tempL = np.linspace(mint.values - 0.1, maxt.values + 0.1, 200)
salL  = np.linspace(mins.values - 0.1, maxs.values + 0.1, 200)
Tg, Sg = np.meshgrid(tempL, salL)

sigma_theta = gsw.sigma0(Sg, Tg)

# ----- Make Figure -----
fig, ax = plt.subplots(figsize=(8,6))

# Density contours
cs = ax.contour(Sg, Tg, sigma_theta, colors='grey', zorder=1, linewidths=0.8)
cl = plt.clabel(cs, fontsize=8, inline=False)

# ----- Scatter by Year -----
# cmap = plt.cm.get_cmap("tab20", len(unique_years))
cmap = plt.cm.get_cmap("jet", len(unique_years))

for i, yr in enumerate(unique_years):
    idx = (years == yr)
    ax.scatter(sal[idx], temp[idx], s=2, color=cmap(i),
               label=str(yr), zorder=3)

# ----- Freezing Temperature Line -----
t_freezing = gsw.t_freezing(salL, p=0, saturation_fraction=0)
ax.plot(salL, t_freezing, color='black', linewidth=1, zorder=2)

# ----- Labels + Decorations -----
ax.set_xlabel('Practical Salinity [g/kg]')
ax.set_ylabel('Potential Temperature [$^\circ$C]')
ax.set_title('Mooring DITN T/S Bottom Sensor (675 m)')

ax.legend(title="Year")
ax.grid(True, linestyle='--', alpha=0.3)
ax.tick_params(direction='out')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import gsw

# Extract variables
temp = DITN_ctd_mid.T
sal  = DITN_ctd_mid.S
time = pd.to_datetime(DITN_ctd_mid.time.values)
years = time.year
unique_years = np.unique(years)

# ----- Density Grid -----
mint = temp.min()
maxt = temp.max()
mins = sal.min()
maxs = sal.max()

tempL = np.linspace(mint.values - 0.1, maxt.values + 0.1, 200)
salL  = np.linspace(mins.values - 0.1, maxs.values + 0.1, 200)
Tg, Sg = np.meshgrid(tempL, salL)

sigma_theta = gsw.sigma0(Sg, Tg)

# ----- Make Figure -----
fig, ax = plt.subplots(figsize=(8,6))

# Density contours
cs = ax.contour(Sg, Tg, sigma_theta, colors='grey', zorder=1, linewidths=0.8)
cl = plt.clabel(cs, fontsize=8, inline=False)

# ----- Scatter by Year -----
# cmap = plt.cm.get_cmap("tab20", len(unique_years))
cmap = plt.cm.get_cmap("jet", len(unique_years))

for i, yr in enumerate(unique_years):
    idx = (years == yr)
    ax.scatter(sal[idx], temp[idx], s=2, color=cmap(i),
               label=str(yr), zorder=3)

# ----- Freezing Temperature Line -----
t_freezing = gsw.t_freezing(salL, p=0, saturation_fraction=0)
ax.plot(salL, t_freezing, color='black', linewidth=1, zorder=2)

# ----- Labels + Decorations -----
ax.set_xlabel('Practical Salinity [g/kg]')
ax.set_ylabel('Potential Temperature [$^\circ$C]')
ax.set_title('Mooring DITN T/S Mid Sensor (275 m)')

ax.legend(title="Year")
ax.grid(True, linestyle='--', alpha=0.3)
ax.tick_params(direction='out')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import gsw

# Extract variables
temp = DITN_ctd_top.T
sal  = DITN_ctd_top.S
time = pd.to_datetime(DITN_ctd_top.time.values)
years = time.year
unique_years = np.unique(years)

# ----- Density Grid -----
mint = temp.min()
maxt = temp.max()
mins = sal.min()
maxs = sal.max()

tempL = np.linspace(mint.values - 0.1, maxt.values + 0.1, 200)
salL  = np.linspace(mins.values - 0.1, maxs.values + 0.1, 200)
Tg, Sg = np.meshgrid(tempL, salL)

sigma_theta = gsw.sigma0(Sg, Tg)

# ----- Make Figure -----
fig, ax = plt.subplots(figsize=(8,6))

# Density contours
cs = ax.contour(Sg, Tg, sigma_theta, colors='grey', zorder=1, linewidths=0.8)
cl = plt.clabel(cs, fontsize=8, inline=False)

# ----- Scatter by Year -----
# cmap = plt.cm.get_cmap("tab20", len(unique_years))
cmap = plt.cm.get_cmap("jet", len(unique_years))

for i, yr in enumerate(unique_years):
    idx = (years == yr)
    ax.scatter(sal[idx], temp[idx], s=2, color=cmap(i),
               label=str(yr), zorder=3)

# ----- Freezing Temperature Line -----
t_freezing = gsw.t_freezing(salL, p=0, saturation_fraction=0)
ax.plot(salL, t_freezing, color='black', linewidth=1, zorder=2)

# ----- Labels + Decorations -----
ax.set_xlabel('Practical Salinity [g/kg]')
ax.set_ylabel('Potential Temperature [$^\circ$C]')
ax.set_title('Mooring DITN T/S Top Sensor (75 m)')

ax.legend(title="Year")
ax.grid(True, linestyle='--', alpha=0.3)
ax.tick_params(direction='out')

plt.tight_layout()
plt.show()

In [ ]:
df.query("mooring == 'DITD'& instrument == 'ctd'")

In [ ]:
# 2 min sampling interval for DITD 2017
# 3 min sampling interval for DITD 2018
# 2 min samplin interval for DITD 2019
# 5 min sampling interval for DITD 2020
# 5 min sampling interval for DITD 2022
# 5 min sampling interval for 2024 

In [ ]:
# USING TOP DITD FOR NOW FOR 2018 
# NEED TO WAIT TO HEAR BACK FROM LIV ABOUT ISSUE WITH BOTTOM CTD SENSOR IN 2018
DITD_2018_ctd_bot = dt['DITD']['2018']['ctd']['top'].to_dataset()
DITD_2018_ctd_bot = DITD_2018_ctd_bot.rename({'DateTime':'time'})

In [ ]:
# DITD_2017_ctd_bot = DITD_2017_ctd_bot.resample(time='5min').mean()
# DITD_2018_ctd_bot = DITD_2018_ctd_bot.resample(time='5min').mean()
# DITD_2019_ctd_bot = DITD_2019_ctd_bot.resample(time='5min').mean()

In [ ]:
# DITD_2018_ctd_bot = DITD_2018_ctd_bot.drop_vars(['absolute_salinity','density','conservative_temp'])

In [ ]:
# DITD_2018_ctd_bot = DITD_2018_ctd_bot.rename({
#     "temp": "T",
#     "practical_salinity":"S",
#     "pressure":"P"
# })

In [ ]:
# DITD_2018_ctd_bot = DITD_2018_ctd_bot.assign_coords(
#     z=1218,
#     lat=-72.28,
#     lon=164.1
# )

In [ ]:
df.query("mooring == 'DITS'")

In [ ]:
DITD_ctd_bot = xr.concat([DITD_2017_ctd_bot,DITD_2018_ctd_bot,DITD_2019_ctd_bot,DITD_2020_ctd_bot,DITD_2022_ctd_bot,DITD_2024_ctd_bot],dim='time')

In [ ]:
DITD_ctd_bot.time[-1]

In [ ]:
# make full time index to fill in with nans where time is missing - aids in plotting later 
full_time = pd.date_range(
    start="2017-02-08T20:20:00",
    end="2024-12-08T08:20:00",
    freq="10min"
)

DITD_ctd_bot = DITD_ctd_bot.sortby("time")     # VERY important after concatenation
DITD_ctd_bot = DITD_ctd_bot.reindex(time=full_time)

In [ ]:
dt['TNBD']['2018']['vel']['top'].lon.values

In [ ]:
fig = plt.figure(figsize=(12,4))
plt.plot(DITD_ctd_bot.time,DITD_ctd_bot.S)
plt.ylabel('Practical Salinity')
plt.title('DITD 1200 m Salinity')

In [ ]:
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(DITD_ctd_bot.time,DITD_ctd_bot.S)
ax.set_ylabel('Practical Salinity')

ax1 = ax.twinx()
ax1.plot(DITD_ctd_bot.time,DITD_ctd_bot.T,color='orange')
ax1.set_ylabel('Potential Temperature [°C]',color='orange')


In [ ]:


# Extract variables
temp = DITD_ctd_bot.T
sal  = DITD_ctd_bot.S
time = pd.to_datetime(DITD_ctd_bot.time.values)
years = time.year
unique_years = np.unique(years)

# ----- Density Grid -----
mint = temp.min()
maxt = temp.max()
mins = sal.min()
maxs = sal.max()

tempL = np.linspace(mint.values - 0.1, maxt.values + 0.1, 200)
salL  = np.linspace(mins.values - 0.1, maxs.values + 0.1, 200)
Tg, Sg = np.meshgrid(tempL, salL)

sigma_theta = gsw.sigma0(Sg, Tg)

# ----- Make Figure -----
fig, ax = plt.subplots(figsize=(8,6))

# Density contours
cs = ax.contour(Sg, Tg, sigma_theta, colors='grey', zorder=1, linewidths=0.8)
cl = plt.clabel(cs, fontsize=8, inline=False)

# ----- Scatter by Year -----
# cmap = plt.cm.get_cmap("tab20", len(unique_years))
cmap = plt.cm.get_cmap("jet", len(unique_years))

for i, yr in enumerate(unique_years):
    idx = (years == yr)
    ax.scatter(sal[idx], temp[idx], s=2, color=cmap(i),
               label=str(yr), zorder=3)

# ----- Freezing Temperature Line -----
t_freezing = gsw.t_freezing(salL, p=0, saturation_fraction=0)
ax.plot(salL, t_freezing, color='black', linewidth=1, zorder=2)

# ----- Labels + Decorations -----
ax.set_xlabel('Practical Salinity [g/kg]')
ax.set_ylabel('Potential Temperature [$^\circ$C]')
ax.set_title('Mooring DITD T/S (1200 m)')

ax.legend(title="Year")
ax.grid(True, linestyle='--', alpha=0.3)
ax.tick_params(direction='out')

plt.tight_layout()
plt.show()

In [ ]:
dt['DITN']['2018']['vel']['top'].u.values

In [ ]:
fig = plt.figure()
dt['DITN']['2018']['vel']['bot'].u.plot()
dt['DITN']['2018']['vel']['top'].u.plot()

In [ ]:
df.query("mooring == 'D'")

In [ ]:
DITN_vel_top = xr.concat([DITN_2017_vel_top,DITN_2018_vel_top,DITN_2019_vel_top,DITN_2020_vel_top],dim='time')

In [ ]:
DITN_vel_mid = xr.concat([DITN_2017_vel_mid,DITN_2018_vel_mid,DITN_2019_vel_mid,DITN_2020_vel_mid],dim='time')

In [ ]:
DITN_vel_bot = xr.concat([DITN_2017_vel_bot,DITN_2018_vel_bot,DITN_2019_vel_bot,DITN_2020_vel_bot],dim='time')

### DITD vs. TNBD 

In [ ]:
fig = plt.figure(figsize=(12,4))
plt.plot(DITD_2024_vel_bot.time,DITD_2024_vel_bot.u)


In [ ]:
# I think velocity units for this year are in m/s not cm/s -> should confirm with Liv 
DITD_2024_vel_bot['u'] = DITD_2024_vel_bot.u *100
DITD_2024_vel_bot['v'] = DITD_2024_vel_bot.v *100

In [ ]:
# DITD_2018_vel_bot = DITD_2018_vel_bot.resample(time='15min').mean()
# DITD_2019_vel_bot = DITD_2019_vel_bot.resample(time='15min').mean()
# DITD_2020_vel_bot = DITD_2020_vel_bot.resample(time='15min').mean()
# DITD_2022_vel_bot = DITD_2022_vel_bot.resample(time='15min').mean()
# DITD_2024_vel_bot = DITD_2024_vel_bot.resample(time='15min').mean()

In [ ]:
DITD_vel_bot = xr.concat([DITD_2017_vel_bot,DITD_2018_vel_bot,DITD_2019_vel_bot,DITD_2020_vel_bot,DITD_2022_vel_bot,DITD_2024_vel_bot],dim='time')

In [ ]:
DITD_vel_bot.time[-1]

In [ ]:
# full_time = pd.date_range(
#     start="2017-02-08T20:30:00",
#     end="2024-12-08T08:10:00",
#     freq="10min"
# )

DITD_vel_bot = DITD_vel_bot.sortby("time")     # VERY important after concatenation
# DITD_vel_bot = DITD_vel_bot.reindex(time=full_time)

In [ ]:
# DITD_2017_vel sampliing freq = 15 min
# 2018 - 10 min 
# 2019 - 10 min 
# 2020 - 10 min 
# 2022 - 10 min 
# 2024 - 10 min 
DITD_2024_vel_bot.time

In [ ]:
fig = plt.figure(figsize=(12,4))
ds = DITN_vel_top
q = plt.quiver(
    ds.time.resample(time='M').mean().values,np.zeros(len(ds.time.resample(time='M').mean())),
    ds.u.resample(time='M').mean(),
    ds.v.resample(time='M').mean(),
    scale=70, units='width'
)
plt.quiverkey(q, X=0.90, Y=0.1, U=5, label='5 cm/s', labelpos='E')

plt.title('DITN 75 m Velocity')
plt.ylabel('Velocity (m/s)')

In [ ]:
fig, ax = plt.subplots(ncols=1,nrows=2,constrained_layout=True,figsize=(12,6))

q = ax[0].quiver(tnbd_vel1000.time.resample(time='W').mean().values,np.zeros(len(tnbd_vel1000.time.resample(time='W').mean().values)),
               tnbd_vel1000.east.resample(time='W').mean(),
               tnbd_vel1000.north.resample(time='W').mean(),
               tnbd_sbe1000.absolute_salinity.resample(time='W').mean(),scale = 0.75,units='width')

q1 = ax[1].quiver(tnbd_vel1100.time.resample(time='W').mean().values,np.zeros(len(tnbd_vel1100.time.resample(time='W').mean().values)),
               tnbd_vel1100.east.resample(time='W').mean(),
               tnbd_vel1100.north.resample(time='W').mean(),
               tnbd_sbe1100.absolute_salinity.resample(time='W').mean(),scale= 0.75, units='width')

plt.colorbar(q,label='Weekly Averaged Absolute Salinity')
# plt.colorbar(q1,label='Weekly Averaged Absolute Salinity')

ax[0].quiverkey(q, X=0.90, Y=0.1, U=0.05, label='0.05 m/s', labelpos='E')

ax[0].set_title('Weekly Averaged Velocity at TNBD 999 m')
ax[1].set_title('Weekly Averaged Velocity at TNBD 1093 m')
# plt.ylim(-2,2)
ax[0].set_xlim(datetime.datetime(2018, 3, 1),datetime.datetime(2019, 2, 10))
ax[1].set_xlim(datetime.datetime(2018, 3, 1),datetime.datetime(2019, 2, 10))
ax[0].hlines(y=0,xmin=datetime.datetime(2018, 3, 1),
           xmax = datetime.datetime(2019, 2, 10),color='grey',linestyle='dashed')

ax[1].hlines(y=0,xmin=datetime.datetime(2018, 3, 1),
           xmax = datetime.datetime(2019, 2, 10),color='grey',linestyle='dashed')
ax[0].set_yticks([])
ax[1].set_yticks([])

In [ ]:
DITN_2018_vel_top_raw = dt['DITN']['2018']['vel']['top'].to_dataset()

In [ ]:
fig = plt.figure(figsize=(12,4))
ds = DITN_2018_vel_top_raw
q = plt.quiver(
    ds.time.resample(time='M').mean().values,np.zeros(len(ds.time.resample(time='M').mean())),
    ds.u.resample(time='M').mean(),
    ds.v.resample(time='M').mean(),
    scale=50, units='width'
)
plt.quiverkey(q, X=0.90, Y=0.1, U=5, label='5 cm/s', labelpos='E')

In [ ]:
DITN_2017_vel_bot

### Time series of TNBD and DITD temp / sal

In [ ]:
TNBD_ctd_bot

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
ds = TNBD_ctd_bot.rolling(time=10).mean().sel(time=slice('2018-03-29','2018-04-01'))
ax.plot(ds.time,ds.practical_salinity,color='tab:blue',label='Salinity')
ax.set_ylabel('Practical Salinity',color='tab:blue')
ax.set_ylim(35.564,35.568)

ax1 = ax.twinx()
ax1.plot(ds.time,ds.temp,color='orange',label='Temperature')
ax1.set_ylabel('Potential Temperature [°C]',color='orange')
ax1.set_ylim(-1.891,-1.888)
# x ticks on angle 
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

# legend
lines, labels = ax.get_legend_handles_labels()
lines1, labels1 = ax1.get_legend_handles_labels()
ax.legend(lines + lines1, labels + labels1, loc='upper right')
